# UAV-VisLoc Benchmark

Runs all feature-matching pipelines on the UAV-VisLoc dataset on Kaggle.

**Code:** cloned from GitHub (`benchmarking` branch) into `/kaggle/working/UAV_Localization`
**Dataset:** Kaggle Dataset mounted read-only at `/kaggle/input/datasets/youssefelsayed30/uav-visloc/`
**Limit:** 100 images per run (change `--limit` in each cell)
**Success threshold:** 25 m GPS error
**Results & visualizations:** saved to `/kaggle/working/results/`

**Run the setup cell first.** After that, each section is independent.

## Setup — clone repo + define shared paths

Run this once at the start of the session.

In [10]:
print('>>> EDIT MARKER v3 — if you see this, edits reached Kaggle <<<')

import os, sys, subprocess, shutil

REPO_URL    = 'https://github.com/YousefAyman005/UAV_Localization.git'
REPO_BRANCH = 'benchmarking'
REPO        = '/kaggle/working/UAV_Localization'
DATA        = '/kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example'
OUT         = '/kaggle/working/results'

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            f'cmd failed ({r.returncode}): {" ".join(cmd)}\n'
            f'STDOUT: {r.stdout}\nSTDERR: {r.stderr}'
        )
    return r

if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    _run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO])
else:
    _run(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    _run(['git', '-C', REPO, 'checkout', REPO_BRANCH])
    _run(['git', '-C', REPO, 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.makedirs(OUT, exist_ok=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')
print(f'OUT:  {OUT}')

>>> EDIT MARKER v3 — if you see this, edits reached Kaggle <<<
REPO: /kaggle/working/UAV_Localization
DATA: /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example
OUT:  /kaggle/working/results


---
## Section 1 — Baseline (OpenCV)

Classical feature detectors using OpenCV. No GPU required, no extra installs.

| Method | Detector | Descriptor | Matcher |
|--------|----------|------------|---------|
| SIFT   | DoG keypoints | 128-d float SIFT | FLANN + Lowe ratio 0.75 |
| ORB    | FAST keypoints | 256-bit binary ORB | BFMatcher Hamming |
| BRISK  | AGAST keypoints | 512-bit binary BRISK | BFMatcher Hamming |

### 1a — Baseline SIFT

Scale-Invariant Feature Transform. Most accurate of the three classical methods. Slower than ORB/BRISK.

In [11]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_sift_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_sift_viz'

sys.argv = ['', '--limit', '100', '--method', 'sift', '--dist', '25', '--visualize']
pl.main()

Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: SIFT | Preprocessing: none | Dist: 25.0m | 100 images



100%|██████████| 100/100 [04:17<00:00,  2.57s/img]


  Results saved to /kaggle/working/results/baseline_sift_results.csv
  Success (≤25.0m):    34/100 (34.0%)
  Homography found:       48/100 (48.0%)
  Incorrect matches:      14/48 (29.2%) — offset > 25.0m
  Offset (successes):     mean 14.9m  median 14.9m  max 23.8m
  Median inliers: 9 | ratio: 0.455


### 1b — Baseline ORB

Oriented FAST and Rotated BRIEF. Very fast binary descriptor. Less accurate than SIFT but runs in real time.

In [ ]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_orb_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_orb_viz'

sys.argv = ['', '--limit', '100', '--method', 'orb', '--dist', '25', '--visualize']
pl.main()

### 1c — Baseline BRISK

Binary Robust Invariant Scalable Keypoints. Similar speed to ORB with a larger 512-bit descriptor. Often more robust to scale changes than ORB.

In [ ]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_brisk_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_brisk_viz'

sys.argv = ['', '--limit', '100', '--method', 'brisk', '--dist', '25', '--visualize']
pl.main()

---
## Section 2 — LightGlue

Learned keypoint matcher that works with multiple front-end detectors. Uses an attention-based GNN to prune ambiguous matches. GPU strongly recommended.

| Variant | Detector | Descriptor | Notes |
|---------|----------|------------|-------|
| DISK    | DISK (learned) | DISK (256-d) | Best accuracy |
| SIFT    | DoG | SIFT (128-d) | Good balance |
| DeDoDe-B | DeDoDe (learned) | DeDoDe (256-d) | Strongest detector |

### 2a — LightGlue + DISK

DISK detector with LightGlue matcher. Typically the strongest LightGlue variant for outdoor aerial imagery.

In [12]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_disk_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_disk_viz'

sys.argv = ['', '--limit', '100', '--method', 'disk',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

  Device: cuda
  Loading models (disk) ... Downloading: "https://raw.githubusercontent.com/cvlab-epfl/disk/master/depth-save.pth" to /root/.cache/torch/hub/checkpoints/depth-save.pth


100%|██████████| 4.17M/4.17M [00:00<00:00, 57.0MB/s]


Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/disk_lightglue.pth" to /root/.cache/torch/hub/checkpoints/disk_lightglue_v0-1_arxiv-pth


100%|██████████| 45.4M/45.4M [00:00<00:00, 183MB/s]


Loaded LightGlue model
done
Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: DISK | CLAHE: False | Conf: 0.0 | RANSAC: 10.0 | MinInl: 6 | Dist: 25.0m | 100 images



100%|██████████| 100/100 [03:02<00:00,  1.83s/img]


  Results saved to /kaggle/working/results/lightglue_disk_results.csv
  Success (≤25.0m):    36/100 (36.0%)
  Homography found:       100/100 (100.0%)
  Incorrect matches:      64/100 (64.0%) — offset > 25.0m
  Offset (successes):     mean 15.8m  median 17.3m  max 25.0m
  Median inliers: 16 | ratio: 0.018


### 2b — LightGlue + SIFT

Classic SIFT detector fed into the LightGlue matcher. Good fallback when DISK/DeDoDe weights are unavailable or slow to download.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_sift_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_sift_viz'

sys.argv = ['', '--limit', '100', '--method', 'sift',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

### 2c — LightGlue + DeDoDe-B

DeDoDe detector (trained to detect repeatable keypoints across viewpoints) with LightGlue. Often finds more matches in low-texture regions than SIFT or DISK.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_dedodeb_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_dedodeb_viz'

sys.argv = ['', '--limit', '100', '--method', 'dedodeb',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

---
## Section 3 — LoFTR

**LoFTR** (Detector-Free Local Feature Matching with Transformers) matches dense pixel pairs directly without detecting keypoints first. Uses a coarse-to-fine Transformer architecture trained on MegaDepth (`outdoor` weights). Strong in low-texture and repetitive regions where keypoint detectors struggle. GPU required for reasonable speed.

### 3a — LoFTR (outdoor)

Pretrained on MegaDepth outdoor scenes. Correct weight set for UAV/satellite aerial matching.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kornia', '-q'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('loftr_pipeline', None)
sys.modules.pop('visloc_utils', None)

import loftr_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/loftr_results.csv'
pl.VIZ_DIR   = f'{OUT}/loftr_viz'

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--dist', '25', '--visualize']
pl.main()

---
## Section 4 — RoMa

**RoMa** (Robust Dense Feature Matching) produces a dense warp field between image pairs using a DINOv2 backbone, then samples correspondence points from it. No keypoint detection step. Typically the most accurate dense matcher for large viewpoint and scale changes. GPU required; slowest of all methods.

Pretrained on MegaDepth (`outdoor`) — correct for aerial/satellite scenes.

### 4a — RoMa (outdoor)

Dense warp-based matching with DINOv2 backbone. Samples 5000 correspondences from the predicted warp field.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/Parskatt/RoMa.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('roma_pipeline', None)
sys.modules.pop('visloc_utils', None)

import roma_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/roma_results.csv'
pl.VIZ_DIR   = f'{OUT}/roma_viz'

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--num-matches', '5000', '--dist', '25', '--visualize']
pl.main()